# EDA - OpenAlex OSS corpus

Exploratory analysis of `data/raw/openalex_works.jsonl` (produced by `ingest/fetch_openalex.py`).

Covers: year distribution, primary topic and keyword frequency, language counts,
abstract length, and the share of works with an abstract.

Outputs CSVs to `data/eda/` (gitignored) and PNG charts to `report/figures/` (committed).
Run with: `jupyter nbconvert --execute eval/eda.ipynb`


In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
if not (REPO / "data").exists() and (REPO.parent / "data").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from ingest.fetch_openalex import API, FILTER, MAILTO, get, load_api_key

DATA = REPO / "data" / "raw" / "openalex_works.jsonl"
EDA_DIR = REPO / "data" / "eda"
FIG_DIR = REPO / "report" / "figures"
EDA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("data:", DATA, "exists:", DATA.exists())


In [ ]:
records = []
with open(DATA, encoding="utf-8") as fh:
    for line in fh:
        w = json.loads(line)
        pt = w.get("primary_topic") or {}
        records.append({
            "id": w["id"],
            "year": w.get("publication_year"),
            "language": w.get("language"),
            "type": w.get("type"),
            "primary_topic": pt.get("display_name"),
            "keywords": [k.get("display_name") for k in (w.get("keywords") or [])],
            "abstract": w.get("abstract"),
        })
df = pd.DataFrame(records)
df["abstract_words"] = df["abstract"].fillna("").str.split().str.len()
print("works:", len(df))
print("with abstract:", int(df["abstract"].notna().sum()),
      f"({df['abstract'].notna().mean():.1%})")


## Year distribution

In [ ]:
years = df["year"].dropna().astype(int)
year_counts = years.value_counts().sort_index()
year_counts.to_csv(EDA_DIR / "year_counts.csv", header=["count"])
print("range:", years.min(), "-", years.max(), "| median:", int(years.median()))
print(year_counts.tail(15).to_string())
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(year_counts.index, year_counts.values)
ax.set_title("Works per publication year")
ax.set_xlabel("year")
ax.set_ylabel("works")
fig.tight_layout()
fig.savefig(FIG_DIR / "year_distribution.png", dpi=120)


## Primary topic and keyword frequency

In [ ]:
pt_counts = df["primary_topic"].value_counts()
pt_counts.head(20).to_csv(EDA_DIR / "primary_topic_top.csv", header=["count"])
kw_top = pd.Series(Counter(k for ks in df["keywords"] for k in ks)).sort_values(ascending=False)
kw_top.head(30).to_csv(EDA_DIR / "keyword_top.csv", header=["count"])
print("distinct primary topics:", pt_counts.size)
print(pt_counts.head(10).to_string())
print("\ntop keywords:")
print(kw_top.head(10).to_string())
fig, ax = plt.subplots(figsize=(10, 6))
pt_counts.head(15)[::-1].plot.barh(ax=ax)
ax.set_title("Top primary topics")
ax.set_xlabel("works")
fig.tight_layout()
fig.savefig(FIG_DIR / "primary_topics.png", dpi=120)


## Language

In [ ]:
lang_counts = df["language"].fillna("(none)").value_counts()
lang_counts.to_csv(EDA_DIR / "language_counts.csv", header=["count"])
print(lang_counts.to_string())


## Abstract length (words)

In [ ]:
n = df.loc[df["abstract"].notna(), "abstract_words"]
summary = pd.Series({
    "works": len(n),
    "mean": n.mean(),
    "median": n.median(),
    "p10": n.quantile(0.10),
    "p90": n.quantile(0.90),
    "p99": n.quantile(0.99),
    "min": n.min(),
    "max": n.max(),
})
summary.to_csv(EDA_DIR / "abstract_length_summary.csv", header=["value"])
hist = n.value_counts(bins=[0, 50, 100, 150, 200, 250, 300, 400, 500, 1000, 10**9]).sort_index()
hist.to_csv(EDA_DIR / "abstract_length_hist.csv", header=["count"])
print(summary.to_string())
print(hist.to_string())
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(n, bins=50)
ax.set_title("Abstract length (words)")
ax.set_xlabel("words")
ax.set_ylabel("works")
fig.tight_layout()
fig.savefig(FIG_DIR / "abstract_length.png", dpi=120)


## API cross-check (topic-wide)

The corpus was fetched with `has_abstract:true` and `language:en`, so in-corpus coverage is 100% and the language is `en` by construction. These queries show the real topic-wide mix.

In [ ]:
def openalex(filter_, group_by=None, per_page=1):
    params = {"filter": filter_, "per-page": per_page, "mailto": MAILTO}
    key = load_api_key()
    if key:
        params["api_key"] = key
    if group_by:
        params["group_by"] = group_by
        params["per-page"] = 200  # per-page caps how many groups come back
    return get(API, params)

total = openalex("topics.id:T11675")["meta"]["count"]
with_abs = openalex("topics.id:T11675,has_abstract:true")["meta"]["count"]
corpus = openalex(FILTER)["meta"]["count"]
cross = pd.DataFrame([
    {"metric": "topic_total", "value": total},
    {"metric": "topic_with_abstract", "value": with_abs},
    {"metric": "corpus_filter", "value": corpus},
    {"metric": "abstract_coverage_of_topic", "value": round(with_abs / total, 4)},
    {"metric": "corpus_share_of_abstracted", "value": round(corpus / with_abs, 4)},
])
cross.to_csv(EDA_DIR / "api_crosscheck.csv", index=False)
print(cross.to_string(index=False))
print(f"\nworks with abstract / topic total = {with_abs / total:.1%}")
print(f"corpus / abstracted topic works = {corpus / with_abs:.1%}")

lang = openalex("topics.id:T11675,has_abstract:true", group_by="language")["group_by"]
lang_df = pd.DataFrame([{"language": g["key_display_name"] or "(none)", "count": g["count"]} for g in lang])
lang_df.to_csv(EDA_DIR / "api_language_group.csv", index=False)
print("\nlanguage mix (topic-wide, top 10):")
print(lang_df.head(10).to_string(index=False))
english = lang_df.loc[lang_df["language"] == "English", "count"].sum()
print(f"english share of abstracted topic works = {english / with_abs:.1%}")

yr = openalex("topics.id:T11675,has_abstract:true", group_by="publication_year")["group_by"]
yr_df = pd.DataFrame([{"year": g["key"], "count": g["count"]} for g in yr]).sort_values("year")
yr_df.to_csv(EDA_DIR / "api_year_group.csv", index=False)


## Notes for BAB 3

- In-corpus metrics are shaped by the fetch filter (article, english, has abstract); the API cross-check gives the honest coverage figure.
- The topic holds many non-English works; restricting to english is a stated limitation.
- Abstract-length outliers and the year skew are corpus limitations worth stating explicitly.
- CSVs in `data/eda/` feed the experiment tables; PNGs in `report/figures/` feed the report.
